In [6]:
import pandas as pd
from pathlib import Path

# Definir os caminhos como objetos Path
arquivo_entrada = Path("06_PENSE2024_SEM_RESPOSTA_ABANDONO_PULO.parquet")
arquivo_saida = Path("07_PENSE2024_IDADE_13_A_17_ANOS.parquet")

# Carregar a base da etapa anterior
df = pd.read_parquet(arquivo_entrada)

# Definir a coluna de idade
coluna_idade = "IDADE DO ESTUDANTE, CATEGORIAS"

# Identificar registros com idade menor que 13 ou maior que 18
mascara_excluir = df[coluna_idade].astype("string").str.strip().isin(["1", "4"])

# Contabilizar registros
total_original = len(df)
total_excluir = mascara_excluir.sum()

# Criar nova base mantendo somente as categorias 2 e 3
df_idade = df.loc[~mascara_excluir].copy()

# Salvar a nova base
df_idade.to_parquet(
    arquivo_saida,
    index=False
)

# Exibir resumo do processamento
print("========== FILTRO DE IDADE CONCLUÍDO ==========")

print(f"Linhas originais: {total_original:,}")
print(f"Linhas removidas: {total_excluir:,}")
print(f"Linhas restantes: {len(df_idade):,}")

print(
    f"Percentual removido: "
    f"{total_excluir / total_original * 100:.2f}%"
)

mapa_idades = {
    "1": "Menos de 13 anos",
    "2": "13 a 14 anos",
    "3": "15 a 17 anos",
    "4": "18 anos ou mais",
    1: "13 a 14 anos",  # Tratamento caso os valores estejam como números inteiros
    2: "13 a 14 anos",
    3: "15 a 17 anos",
    4: "18 anos ou mais",
}

print("\nDistribuição das categorias de idade após o filtro:")

# Mapeia os valores mantendo a contagem e a ordenação
distribuicao = df_idade[coluna_idade].map(mapa_idades)
print(distribuicao.value_counts(dropna=False).sort_index())

print("\nArquivo salvo em:")
print(arquivo_saida)

print("\nArquivo existe:", arquivo_saida.exists())

# Definir o caminho completo do novo arquivo
arquivo_saida = Path("07_PENSE2024_IDADE_13_A_17_ANOS.parquet")

# Salvar a nova base em formato Parquet
df_idade.to_parquet(
    arquivo_saida,
    index=False
)

# Confirmar a gravação
print("Arquivo gravado com sucesso!")
print(f"Localização: {arquivo_saida}")
print(f"Total de registros: {len(df_idade):,}")
print(f"Arquivo existe: {arquivo_saida.exists()}")

========== FILTRO DE IDADE CONCLUÍDO ==========
Linhas originais: 6,633
Linhas removidas: 841
Linhas restantes: 5,792
Percentual removido: 12.68%

Distribuição das categorias de idade após o filtro:
IDADE DO ESTUDANTE, CATEGORIAS
13 a 14 anos    2314
15 a 17 anos    3478
Name: count, dtype: int64

Arquivo salvo em:
07_PENSE2024_IDADE_13_A_17_ANOS.parquet

Arquivo existe: True
Arquivo gravado com sucesso!
Localização: 07_PENSE2024_IDADE_13_A_17_ANOS.parquet
Total de registros: 5,792
Arquivo existe: True


In [5]:
import pandas as pd
from IPython.display import HTML, display

# 1. Carregar o dicionário de dados
dicionario = pd.read_excel("00_Dicionário_Microdados_PENSE2024.xlsx")

dico_map = {}
for _, row in dicionario.iterrows():
    desc = str(row["Descrição da Variável"]).strip()
    cat = str(row["Categoria"]) if pd.notna(row["Categoria"]) else ""
    cat_dict = {}
    if cat:
        for linha in cat.split("\n"):
            if " - " in linha:
                partes = linha.split(" - ", 1)
                cat_dict[partes[0].strip()] = partes[1].strip()
    dico_map[desc] = cat_dict

# 2. Gerar a tabela resumo para df_idade
tabela_dados = []

for col in df_idade.columns:
    counts = df_idade[col].astype(str).str.strip().value_counts(dropna=False)
    total = len(df_idade)
    cat_dict = dico_map.get(col, {})
    
    respostas_formatadas = []
    for cod, qtd in counts.items():
        rotulo = cat_dict.get(cod, "")
        pct = (qtd / total) * 100
        if rotulo:
            respostas_formatadas.append(f"[{cod}] {rotulo}: {pct:.2f}%")
        else:
            respostas_formatadas.append(f"[{cod}]: {pct:.2f}%")
            
    tabela_dados.append({
        "Pergunta / Variável": col,
        "Distribuição das Respostas": "\n".join(respostas_formatadas)
    })

df_tabela = pd.DataFrame(tabela_dados)

# 3. Estilizar e exibir
styler = df_tabela.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left'
})

print("========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========\n")
display(HTML(styler.to_html()))

========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========



,Pergunta / Variável,Distribuição das Respostas
0,"NOS ÚLTIMOS 12 MESES, alguma vez você sentiu vontade de se machucar de propósito?",[1] Sim: 52.69% [2] Não: 47.31%
1,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que a vida não vale a pena ser vivida?",[1] Nunca: 29.45% [3] Às vezes: 19.85% [5] Sempre: 18.59% [2] Raramente: 17.85% [4] Na maioria das vezes: 14.24%
2,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu triste?",[3] Às vezes: 31.65% [4] Na maioria das vezes: 24.78% [5] Sempre: 18.42% [2] Raramente: 16.42% [1] Nunca: 8.74%
3,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que ninguém se preocupa com você?",[3] Às vezes: 25.76% [2] Raramente: 19.67% [5] Sempre: 18.82% [4] Na maioria das vezes: 18.40% [1] Nunca: 17.35%
4,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu muito preocupado com as coisas comuns do seu dia a dia como atividades da escola, competições esportivas, tarefas de casa, etc.?",[5] Sempre: 26.74% [4] Na maioria das vezes: 26.73% [3] Às vezes: 20.77% [1] Nunca: 12.98% [2] Raramente: 12.78%
5,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu irritado(a), nervoso(a) ou mal-humorado(a) por qualquer coisa?",[5] Sempre: 30.59% [4] Na maioria das vezes: 27.76% [3] Às vezes: 24.78% [2] Raramente: 10.05% [1] Nunca: 6.82%
6,Quantos(as) amigos(as) próximos você tem?,[4] 3 ou mais amigos: 65.54% [3] 2 amigos: 19.15% [2] 1 amigo: 9.06% [1] Nenhum amigo: 6.25%
7,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável entendeu suas preocupações e problemas?",[1] Nunca: 30.23% [2] Raramente: 22.17% [3] Às vezes: 19.79% [4] Na maioria das vezes: 15.44% [5] Sempre: 12.38%
8,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre?",[4] Na maioria das vezes: 30.58% [5] Sempre: 21.27% [3] Às vezes: 18.49% [2] Raramente: 15.37% [1] Nunca: 14.30%
9,"NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola o esculachou, zoou, mangou, intimidou ou caçoou tanto que você ficou magoado, incomodado, aborrecido, ofendido ou humilhado?",[1] Nenhuma vez nos últimos 30 dias: 54.77% [4] 3 ou mais vezes: 22.93% [2] 1 vez: 12.48% [3] 2 vezes: 9.82%
